In [ ]:
import numpy as np 
import matplotlib.pyplot as plt 
from astropy.io import fits
from astropy.table import Table
from astropy.table import vstack
import os

import jax
import seaborn as sns
import pickle

import matplotlib as mpl

mpl.rcParams.update({
    "font.family": "serif",
    "font.serif": ["TeX Gyre Pagella", "Book Antiqua", "Palatino Linotype", "DejaVu Serif"]
})

### Import data

In [ ]:
fits_file_2384a = '/projects/mccleary_group/habjan.e/SuperBIT/data/Abell2384a_colors_mags.fits'
hdul_2384a = fits.open(fits_file_2384a)
data_2384a = Table(hdul_2384a[1].data)

fits_file_2384b = '/projects/mccleary_group/habjan.e/SuperBIT/data/Abell2384b_colors_mags.fits'
hdul_2384b = fits.open(fits_file_2384b)
data_2384b = Table(hdul_2384b[1].data)

fits_file_3667 = '/projects/mccleary_group/habjan.e/SuperBIT/data/Abell3667_colors_mags.fits'
hdul_3667 = fits.open(fits_file_3667)
data_3667 = Table(hdul_3667[1].data)

fits_file_3571 = '/projects/mccleary_group/habjan.e/SuperBIT/data/Abell3571_colors_mags.fits'
hdul_3571 = fits.open(fits_file_3571)
data_3571 = Table(hdul_3571[1].data)

fits_file_3827 = '/projects/mccleary_group/habjan.e/SuperBIT/data/Abell3827_colors_mags.fits'
hdul_3827 = fits.open(fits_file_3827)
data_3827 = Table(hdul_3827[1].data)

fits_file_cosmos113 = '/projects/mccleary_group/habjan.e/SuperBIT/data/COSMOS113_colors_mags.fits'
hdul_cosmos113 = fits.open(fits_file_cosmos113)
data_cosmos113 = Table(hdul_cosmos113[1].data)

fits_file_1689 = '/projects/mccleary_group/saha/data/Abell1689/sextractor_dualmode/out/Abell1689_colors_mags.fits'
hdul_1689 = fits.open(fits_file_1689)
data_1689 = Table(hdul_1689[1].data)

data = vstack([data_2384a, data_2384b, data_3667, data_3571, data_3827])

redshift_col = 'Z_lovoccs'
data_z = data[~np.isnan(np.array(data[redshift_col])) & (np.array(data[redshift_col]) > 0)]
x_cols = ['m_b', 'm_g', 'm_u', 'R_b', 'R_g',  'R_u']

cosmos_col = 'Z_desi'
data_cosmos = data_cosmos113[~np.isnan(np.array(data_cosmos113[cosmos_col])) & (np.array(data_cosmos113[cosmos_col]) > 0)]

### Visualize Vignettes

In [ ]:
s = ''

res_rows = [
    ("Bright Source", s, 61),
    ("Moderately Bright",  s, 54),
    ("Dim Source",  s, 52),
]
bands = ["u", "b", "g"]

q_low, q_high = 0.2, 0.99 


all_imgs = []
for _, suffix, obj in res_rows:
    for band in bands:
        all_imgs.append(np.log10(np.asarray(data_z[f"VIGNET_{band}{suffix}"][obj])))
stack_all = np.stack(all_imgs, axis=0)
vmin = np.nanquantile(stack_all, q_low)
vmax = np.nanquantile(stack_all, q_high)

# --- plot ---
fig, axs = plt.subplots(
    nrows=len(res_rows), ncols=len(bands),
    figsize=(12, 10),
    sharey="row",
    gridspec_kw={"wspace": 0.02, "hspace": 0.08},
)

last_im = None
for r, (row_label, suffix, obj) in enumerate(res_rows):
    for c, band in enumerate(bands):
        ax = axs[r, c]
        img = data_z[f"VIGNET_{band}{suffix}"][obj]
        image = np.log10(img)
        image[np.isnan(image)] = np.nanmin(image)

        last_im = ax.imshow(
            image, vmin=vmin, vmax=vmax, cmap="cubehelix",
            origin="lower", interpolation="nearest"
        )

        if r == 0:
            ax.set_title(f"{band}-band", fontsize=20, pad=6)
        if c == 0:
            #ax.set_ylabel(row_label, fontsize=12)
            z, z_err = np.round(np.array(data_z['Z_lovoccs'])[obj], 2), np.round(np.array(data_z['ZERR_lovoccs'])[obj], 2)
            ax.set_ylabel(f'z = {z}'r'$\pm$'f'{z_err}', fontsize=20)

        ax.set_xticks([])
        ax.set_yticks([])
        for spine in ax.spines.values():
            spine.set_visible(False)


cbar = fig.colorbar(last_im, ax=axs, fraction=0.03, pad=0.02)
cbar.set_label(r"Pixel Brightness [$log_{10}($ Photon $/$ pixel$)$]", fontsize=20)
cbar.ax.tick_params(labelsize=12)

fig.savefig("/home/habjan.e/SuperBIT_code/Redshift_ml/ml_redshifts/figures/superbit_objects.png", bbox_inches="tight", dpi=300)

### Import true and sampled data

In [ ]:
### Trained run to load
run_name = 'censor'
data_path = '/projects/mccleary_group/habjan.e/SuperBIT/data/model_datasets/'
model_path = '/projects/mccleary_group/habjan.e/SuperBIT/data/model_files/'

splits = ['train', 'validation', 'test']

### Posterior samples and the matching true redshifts, written by model_infer.py.
### Rows are in file order, so samples[i] and z_true[i] are the same galaxy.
### Each npz also carries z_pred_mean/median/sigma/p16/p84 and the provenance
### columns (ra, dec, field, group_id).
pred = {}
for split in splits:
    with np.load(data_path + f'{run_name}_{split}_predictions.npz',
                 allow_pickle=True) as f:
        pred[split] = {k: f[k] for k in f.files}

### Loss curves. train_loss is one value per step, val/test one per eval_every
### steps, so they need separate x axes.
train_loss = np.load(model_path + f'{run_name}_train_loss.npy')
val_loss = np.load(model_path + f'{run_name}_val_loss.npy')
test_loss = np.load(model_path + f'{run_name}_test_loss.npy')
eval_every = len(train_loss) // len(val_loss)

with open(model_path + f'{run_name}_metrics.pkl', 'rb') as f:
    val_metrics = pickle.load(f)
with open(model_path + f'{run_name}_test_metrics.pkl', 'rb') as f:
    test_metrics = pickle.load(f)


def photoz_stats(p):
    """Standard photo-z metrics on dz = (z_pred - z_true) / (1 + z_true)."""
    z_pred, z_true = p['z_pred_mean'], p['z_true']
    dz = (z_pred - z_true) / (1.0 + z_true)
    bias = np.median(dz)
    return dict(
        N=len(z_true),
        bias=bias,
        nmad=1.4826 * np.median(np.abs(dz - bias)),
        outlier=np.mean(np.abs(dz) > 0.15),
        rmse=np.sqrt(np.mean((z_pred - z_true) ** 2)),
        coverage68=np.mean((z_true >= p['z_pred_p16']) & (z_true <= p['z_pred_p84'])),
    )


stats = {s: photoz_stats(pred[s]) for s in splits}

### Table view backing the figure below
print(f"run '{run_name}'  |  {pred['train']['samples'].shape[1]} posterior draws "
      f"per galaxy  |  trained {len(train_loss)} steps")
print(f"\n{'split':<12}{'truth':<12}{'N':>6}{'NMAD':>9}{'outlier':>9}"
      f"{'bias':>9}{'RMSE':>8}{'cov68':>8}")
for s in splits:
    v = stats[s]
    print(f"{s:<12}{str(pred[s]['z_source']):<12}{v['N']:>6}{v['nmad']:>9.4f}"
          f"{100 * v['outlier']:>8.1f}%{v['bias']:>+9.4f}{v['rmse']:>8.4f}"
          f"{100 * v['coverage68']:>7.1f}%")

### Make a scatter plot of the jax classification

In [ ]:
### Categorical slots 1-3 of the validated palette; these three clear the
### all-pairs colourblind and normal-vision floors, which is what a scatter
### needs (adjacent-pair validation only covers bars and lines).
COLORS = {'train': '#2a78d6', 'validation': '#eb6834', 'test': '#1baf7a'}
LABELS = {'train': 'Train (LoVoCCS BPZ)',
          'validation': 'Validation (LoVoCCS BPZ)',
          'test': 'Test (DESI spec-z)'}
GRID = '#d8d7d2'
INK, INK2 = '#0b0b0b', '#52514e'

fig, axs = plt.subplots(2, 2, figsize=(11.5, 9),
                        gridspec_kw={'wspace': 0.26, 'hspace': 0.3})
zlim = (0.0, 1.8)

### One panel per split rather than three overlapping clouds: 8735 training
### points would bury the 655 test points entirely.
for ax, split in zip(axs.flat[:3], splits):
    p, c, v = pred[split], COLORS[split], stats[split]
    z_true, z_pred = p['z_true'], p['z_pred_mean']

    ax.plot(zlim, zlim, color=INK2, lw=1, zorder=1)

    ### Individual galaxies, then the binned median with its 16-84 spread so
    ### the trend is readable through the overplotting
    ax.scatter(z_true, z_pred, s=5, c=c, alpha=0.18, linewidths=0, zorder=2)

    edges = np.linspace(0.0, 1.5, 11)
    idx = np.digitize(z_true, edges) - 1
    mids, med, lo, hi = [], [], [], []
    for b in range(len(edges) - 1):
        m = idx == b
        if m.sum() < 15:
            continue
        q16, q50, q84 = np.percentile(z_pred[m], [16, 50, 84])
        mids.append(0.5 * (edges[b] + edges[b + 1]))
        med.append(q50); lo.append(q50 - q16); hi.append(q84 - q50)
    ax.errorbar(mids, med, yerr=[lo, hi], fmt='o', ms=5, lw=1.6, capsize=3,
                color=c, mec='#fcfcfb', mew=1.0, zorder=3)

    ### Direct label: the aqua slot sits below 3:1 on a light surface, so every
    ### panel names its own series instead of relying on colour alone
    ax.set_title(LABELS[split], fontsize=12, color=INK, pad=8)
    ax.text(0.04, 0.955,
            f"N = {v['N']}\n",
            #r"$\sigma_{\rm NMAD}$ = " + f"{v['nmad']:.3f}\n"
            #f"outliers = {100 * v['outlier']:.1f}%\n"
            #f"68% cov = {100 * v['coverage68']:.0f}%",
            transform=ax.transAxes, va='top', ha='left', fontsize=9.5,
            color=INK2, linespacing=1.5)

    ax.set_xlim(*zlim); ax.set_ylim(*zlim)
    ax.set_xlabel('True redshift', fontsize=12)
    ax.set_ylabel('Sampled redshift (posterior mean)', fontsize=12)

### Loss curves. Train is one point per step and very noisy, so the raw trace
### is drawn faintly under a running mean.
ax = axs.flat[3]
steps = np.arange(1, len(train_loss) + 1)
w = 100
smooth = np.convolve(train_loss, np.ones(w) / w, mode='valid')
ax.plot(steps, train_loss, color=COLORS['train'], lw=0.7, alpha=0.13, zorder=1)
ax.plot(steps[w - 1:], smooth, color=COLORS['train'], lw=1.8,
        label=f'Train (running mean, {w} steps)', zorder=3)

eval_steps = (np.arange(len(val_loss)) + 1) * eval_every
ax.plot(eval_steps, val_loss, color=COLORS['validation'], lw=1.8,
        label='Validation', zorder=4)
ax.plot(eval_steps, test_loss, color=COLORS['test'], lw=1.8,
        label='Test (monitored, never selected on)', zorder=5)

#best = min(val_metrics, key=lambda m: m['nmad'])['step']
#ax.axvline(best, color=INK2, lw=1, zorder=2)
#ax.annotate(f'best $\\sigma_{{\\rm NMAD}}$, step {best}', xy=(best, 0.02),
 #           xycoords=('data', 'axes fraction'), xytext=(6, 0),
  #          textcoords='offset points', va='bottom', ha='left',
   #         fontsize=9.5, color=INK2)

### Scale to the curves that matter; the raw per-step trace is noise around them
curves = np.concatenate([smooth, val_loss, test_loss])
ax.set_ylim(curves.min() - 0.15, curves.max() + 0.55)

ax.set_title('Flow-matching loss', fontsize=12, color=INK, pad=8)
ax.set_xlabel('Training step', fontsize=12)
ax.set_ylabel('Flow-matching MSE', fontsize=12)
ax.legend(loc='upper right', fontsize=9.5, frameon=False)

for ax in axs.flat:
    ax.grid(True, color=GRID, lw=0.6, zorder=0)
    ax.set_axisbelow(True)
    ax.tick_params(colors=INK2, labelsize=10)
    for side in ('top', 'right'):
        ax.spines[side].set_visible(False)
    for side in ('left', 'bottom'):
        ax.spines[side].set_color(GRID)

fig_dir = '/home/habjan.e/SuperBIT_code/Redshift_ml/ml_redshifts/figures'
fig.savefig(os.path.join(fig_dir, f'true_vs_sampled.png'),
            dpi=200, bbox_inches='tight', facecolor='white')
plt.show()